# Prostate Cancer Risk Prediction

**Goal.** Explore the `prostate_cancer_prediction.csv` dataset, understand which recorded
clinical/lifestyle factors relate to a `Biopsy_Result` (Benign vs. Malignant), and train
classification models to predict that outcome.

**Contents**
1. [Setup & Data Loading](#1.-Setup-&-Data-Loading)
2. [Data Cleaning](#2.-Data-Cleaning)
3. [Exploratory Data Analysis (EDA)](#3.-Exploratory-Data-Analysis-(EDA))
4. [Bivariate Analysis vs. Biopsy Result](#4.-Bivariate-Analysis-vs.-Biopsy-Result)
5. [Feature Preparation](#5.-Feature-Preparation)
6. [Baseline Classification Models](#6.-Baseline-Classification-Models)
7. [Handling Class Imbalance (SMOTE vs. Undersampling)](#7.-Handling-Class-Imbalance-(SMOTE-vs.-Undersampling))
8. [Feature Engineering & Feature Selection](#8.-Feature-Engineering-&-Feature-Selection)
9. [Hyperparameter Tuning (XGBoost)](#9.-Hyperparameter-Tuning-(XGBoost))
10. [Final Comparison & Conclusion](#10.-Final-Comparison-&-Conclusion)

> **Note on this revision.** This notebook is a cleaned-up, English-only version of the
> original draft. Duplicate cells, dead code, and re-loaded/re-encoded copies of the data
> were removed and replaced with a single linear pipeline so the notebook can be run
> top-to-bottom without `NameError`s. See the last section for a list of the bugs that were
> fixed and why.


## 1. Setup & Data Loading

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, chi2_contingency

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

DATA_PATH = "../data/prostate_cancer_prediction.csv"  # adjust if running outside notebooks/
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()


## 2. Data Cleaning

The dataset has no missing values, but the imputation step below is kept as a defensive
check: it only fills numerical columns with the median and categorical columns with the
mode when a column actually contains missing values, so nothing changes on this dataset
but the notebook stays safe to reuse on a version of the file that does have gaps.


In [ ]:
print("Missing values per column (before handling):")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() else "None found.")

numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
numerical_cols.remove("Patient_ID")  # identifier, not a measurement
categorical_cols = df.select_dtypes(include="object").columns.tolist()

for col in numerical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(f"\nNumerical columns ({len(numerical_cols)}): {numerical_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")


## 3. Exploratory Data Analysis (EDA)

### 3.1 Numerical variables
Histogram + box plot for every numerical column (excluding the `Patient_ID` identifier).


In [ ]:
for col in numerical_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(df[col], kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Frequency")

    sns.boxplot(y=df[col], ax=axes[1])
    axes[1].set_title(f"Box plot of {col}")
    axes[1].set_ylabel(col)

    plt.tight_layout()
    plt.show()


### 3.2 Categorical variables
Count plot for every categorical column except the target (`Biopsy_Result`, shown on its own
right after).


In [ ]:
for col in categorical_cols:
    if col == "Biopsy_Result":
        continue
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=col, hue=col, palette="viridis", legend=False,
                  order=df[col].value_counts().index)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


### 3.3 Target variable — `Biopsy_Result`

In [ ]:
counts = df["Biopsy_Result"].value_counts()
ratio = counts / counts.sum()

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Biopsy_Result", hue="Biopsy_Result", palette="viridis", legend=False)
plt.title("Distribution of Biopsy_Result")
plt.xlabel("Biopsy Result")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print(counts)
print(f"\nClass balance: {ratio['Benign']:.1%} Benign vs. {ratio['Malignant']:.1%} Malignant")
print("-> The target is moderately imbalanced (roughly 7:3), which is addressed in Section 7.")


**EDA summary.** `PSA_Level` and `Prostate_Volume` are right-skewed with a long tail of high
values; `Age`, `Screening_Age`, and `BMI` are roughly symmetric with no extreme outliers.
Most categorical clinical-symptom flags (`Difficulty_Urinating`, `Blood_in_Urine`, etc.) are
imbalanced toward "No", which is expected for symptom indicators. `Biopsy_Result` itself is
moderately imbalanced (~70% Benign / ~30% Malignant).


## 4. Bivariate Analysis vs. Biopsy Result

### 4.1 Numerical features vs. Biopsy Result
Box/violin plots plus a Mann-Whitney U test (no normality assumption needed) for each
numerical feature, comparing the Benign and Malignant groups.


In [ ]:
mannwhitney_results = []

for col in numerical_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.boxplot(x="Biopsy_Result", y=col, data=df, hue="Biopsy_Result", legend=False, ax=axes[0])
    axes[0].set_title(f"{col} by Biopsy Result (box)")

    sns.violinplot(x="Biopsy_Result", y=col, data=df, hue="Biopsy_Result", legend=False, ax=axes[1])
    axes[1].set_title(f"{col} by Biopsy Result (violin)")

    plt.tight_layout()
    plt.show()

    benign_vals = df.loc[df["Biopsy_Result"] == "Benign", col]
    malignant_vals = df.loc[df["Biopsy_Result"] == "Malignant", col]
    stat, p_value = mannwhitneyu(benign_vals, malignant_vals, alternative="two-sided")
    mannwhitney_results.append({"Feature": col, "U statistic": stat, "p-value": p_value})

mannwhitney_df = pd.DataFrame(mannwhitney_results).sort_values("p-value")
mannwhitney_df


### 4.2 Categorical features vs. Biopsy Result
Count plots plus a Chi-squared test of independence for each categorical feature.

In [ ]:
chi2_results = []

for col in categorical_cols:
    if col == "Biopsy_Result":
        continue

    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=col, hue="Biopsy_Result", palette="viridis")
    plt.title(f"{col} by Biopsy Result")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Biopsy Result")
    plt.tight_layout()
    plt.show()

    contingency = pd.crosstab(df[col], df["Biopsy_Result"])
    chi2, p_value, dof, _ = chi2_contingency(contingency)
    chi2_results.append({"Feature": col, "Chi2 statistic": chi2, "dof": dof, "p-value": p_value})

chi2_df = pd.DataFrame(chi2_results).sort_values("p-value")
chi2_df


**Bivariate summary — an important caveat.** At a 5% significance threshold, essentially
none of the numerical or categorical features show a statistically significant association
with `Biopsy_Result` (all p-values are large; group means/medians and category-wise
proportions are nearly identical between Benign and Malignant). For example, mean
`PSA_Level` is ~7.76 for Benign vs. ~7.74 for Malignant, and every `Cancer_Stage` category
splits ~70/30 Benign/Malignant — matching the overall base rate rather than a real signal.

This suggests `Biopsy_Result` in this file was generated largely **independently** of the
other columns (a common trait of synthetic/practice datasets). That matters for Section 10:
no amount of modeling can recover a signal that isn't in the data, so the classification
metrics below should be read as a **methodology demonstration**, not as clinically
meaningful predictive performance.


## 5. Feature Preparation

Before encoding, a few columns are dropped:

- `Patient_ID` — identifier, carries no predictive information.
- `Cancer_Stage`, `Treatment_Recommended`, `Survival_5_Years`, `Follow_Up_Required` — these
  are clinically only known **after** a Malignant biopsy result (stage, treatment plan,
  follow-up need, survival outlook). Keeping them as predictors of `Biopsy_Result` would be
  target leakage in a real clinical workflow, even though in this particular file they
  happen to carry no extra statistical signal either way (Section 4).

The remaining categorical features are one-hot encoded, and the target is mapped to a
binary label (`Malignant` = 1) used throughout the rest of the notebook.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

LEAKAGE_COLS = ["Cancer_Stage", "Treatment_Recommended", "Survival_5_Years", "Follow_Up_Required"]

X = df.drop(columns=["Biopsy_Result", "Patient_ID"] + LEAKAGE_COLS)
y = df["Biopsy_Result"].map({"Benign": 0, "Malignant": 1})

categorical_features = X.select_dtypes(include="object").columns

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(X[categorical_features])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical_features), index=X.index)

X = pd.concat([X.drop(columns=categorical_features), encoded_df], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Training class balance:\n{y_train.value_counts(normalize=True)}")


## 6. Baseline Classification Models

Five classifiers are trained on standardized features: Logistic Regression, Decision Tree,
Random Forest, SVC, and K-Nearest Neighbors. `y_train`/`y_test` are already the binary label
(1 = Malignant), so metrics are computed directly without any string-to-binary conversion.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)


def build_models():
    """Fresh, unfitted model instances (needed each time we retrain on resampled data)."""
    return {
        "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, solver="liblinear"),
        "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
        "SVC": SVC(random_state=RANDOM_STATE, probability=True),
        "K-Nearest Neighbors": KNeighborsClassifier(),
    }


def evaluate(models, X_tr, y_tr, X_te, y_te):
    """Fit each model and return a dict of accuracy/precision/recall/F1/ROC-AUC."""
    results = {}
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        metrics = {
            "Accuracy": accuracy_score(y_te, y_pred),
            "Precision": precision_score(y_te, y_pred, zero_division=0),
            "Recall": recall_score(y_te, y_pred, zero_division=0),
            "F1-Score": f1_score(y_te, y_pred, zero_division=0),
        }
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_te)[:, 1]
            metrics["ROC-AUC"] = roc_auc_score(y_te, y_proba)
        else:
            metrics["ROC-AUC"] = None
        results[name] = metrics
    return results


models = build_models()
results = evaluate(models, X_train_scaled, y_train, X_test_scaled, y_test)
results_df = pd.DataFrame(results).T
results_df


> **Bug fixed vs. the original draft.** The original code assumed `predict_proba(...)[:, 1]`
> always corresponds to the "Malignant" class, which only happened to be true because
> `LabelEncoder`/`sklearn` sorts classes alphabetically (`Benign` < `Malignant`). Fitting on
> the pre-mapped binary `y_train` (1 = Malignant) removes that fragile assumption entirely —
> column 1 of `predict_proba` is guaranteed to be the probability of class `1`.


In [ ]:
def plot_metrics(df_results, metric_name):
    plt.figure(figsize=(8, 4))
    sns.barplot(x=df_results.index, y=metric_name, data=df_results, hue=df_results.index, legend=False)
    plt.title(f"{metric_name} by Model")
    plt.xlabel("Model")
    plt.ylabel(metric_name)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


for metric in ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]:
    plot_metrics(results_df, metric)


## 7. Handling Class Imbalance (SMOTE vs. Undersampling)

Baseline recall on the Malignant class is typically low because the classifiers default to
predicting the majority class (Benign). Both resampling strategies are applied **only to the
training fold** (never to the test set) to avoid data leakage, and both reuse the exact same
scaled train/test split from Section 6 instead of reloading and re-encoding the CSV from
scratch (as the original draft did).


In [ ]:
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"Class balance before SMOTE: {Counter(y_train)}")
print(f"Class balance after SMOTE:  {Counter(y_train_smote)}")

rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_under, y_train_under = rus.fit_resample(X_train_scaled, y_train)
print(f"\nClass balance after undersampling: {Counter(y_train_under)}")


In [ ]:
results_smote = evaluate(build_models(), X_train_smote, y_train_smote, X_test_scaled, y_test)
results_under = evaluate(build_models(), X_train_under, y_train_under, X_test_scaled, y_test)

results_smote_df = pd.DataFrame(results_smote).T
results_under_df = pd.DataFrame(results_under).T

print("--- With SMOTE (oversampling) ---")
display(results_smote_df)
print("\n--- With Random Undersampling ---")
display(results_under_df)


In [ ]:
comparison = []
for name in models:
    comparison.append({"Model": name, "Recall": results[name]["Recall"], "Method": "Original"})
    comparison.append({"Model": name, "Recall": results_smote[name]["Recall"], "Method": "SMOTE (Over)"})
    comparison.append({"Model": name, "Recall": results_under[name]["Recall"], "Method": "Undersampling"})

comparison_df = pd.DataFrame(comparison)

plt.figure(figsize=(12, 5))
sns.barplot(x="Model", y="Recall", hue="Method", data=comparison_df, palette="viridis")
plt.title("Recall Comparison: Original vs. SMOTE vs. Undersampling")
plt.ylabel("Recall (Sensitivity for Malignant)")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Balancing Method", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


**Takeaway.** Both resampling strategies substantially raise recall on the Malignant class
(fewer missed cases) at some cost to precision/accuracy — the expected trade-off of
correcting class imbalance. SMOTE tends to be the more stable choice here since it
synthesizes new minority examples rather than discarding majority-class data.


## 8. Feature Engineering & Feature Selection

New features capture nonlinear/interaction effects that a linear or tree model might not
find on its own: a log transform for the two right-skewed variables, PSA-based clinical
ratios, an age x BMI interaction, and a binned age group. `RFE` (Recursive Feature
Elimination) with a Random Forest then narrows the enlarged feature set down to the 15 most
informative columns.


In [ ]:
df_eng = df.copy()
df_eng["Log_PSA"] = np.log1p(df_eng["PSA_Level"])
df_eng["Log_Prostate_Volume"] = np.log1p(df_eng["Prostate_Volume"])
df_eng["PSA_to_Age_Ratio"] = df_eng["PSA_Level"] / (df_eng["Age"] + 1)
df_eng["PSA_to_Volume_Ratio"] = df_eng["PSA_Level"] / (df_eng["Prostate_Volume"] + 1)
df_eng["BMI_Age_Interaction"] = df_eng["BMI"] * df_eng["Age"]
df_eng["Age_Group"] = pd.cut(df_eng["Age"], bins=[0, 50, 70, 100], labels=["Young", "Middle", "Senior"])

print("New engineered columns:")
display(df_eng[["Log_PSA", "PSA_to_Age_Ratio", "PSA_to_Volume_Ratio", "BMI_Age_Interaction", "Age_Group"]].head())


In [ ]:
X_eng = df_eng.drop(columns=["Biopsy_Result", "Patient_ID"] + LEAKAGE_COLS)
y_eng = df_eng["Biopsy_Result"].map({"Benign": 0, "Malignant": 1})

cat_cols_eng = X_eng.select_dtypes(include=["object", "category"]).columns
X_eng = pd.get_dummies(X_eng, columns=cat_cols_eng, drop_first=True)

X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_eng, y_eng, test_size=0.2, random_state=RANDOM_STATE, stratify=y_eng
)

print(f"Engineered feature matrix: {X_eng.shape}")


In [ ]:
from sklearn.feature_selection import RFE

base_model = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)
selector = RFE(estimator=base_model, n_features_to_select=15, step=1)
selector.fit(X_train_e, y_train_e)

selected_features = X_train_e.columns[selector.support_]
print("Top 15 features selected by RFE:")
for i, feat in enumerate(selected_features, start=1):
    print(f"{i:>2}. {feat}")

X_train_selected = X_train_e[selected_features]
X_test_selected = X_test_e[selected_features]


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline

engineered_pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=RANDOM_STATE)),
])
engineered_pipeline.fit(X_train_selected, y_train_e)

y_pred_eng = engineered_pipeline.predict(X_test_selected)
y_proba_eng = engineered_pipeline.predict_proba(X_test_selected)[:, 1]

print(f"Engineered + selected features -> ROC-AUC: {roc_auc_score(y_test_e, y_proba_eng):.4f}")
print(f"Engineered + selected features -> Recall:  {recall_score(y_test_e, y_pred_eng):.4f}")


## 9. Hyperparameter Tuning (XGBoost)

A `SMOTE` + `XGBoost` pipeline is tuned with `RandomizedSearchCV` (3-fold CV, scoring on
ROC-AUC) over the RFE-selected features.

> **Bug fixed vs. the original draft.** `use_label_encoder=False` was a transitional
> `XGBClassifier` argument for pre-1.6 releases; recent `xgboost` versions removed it
> entirely and raise a `TypeError` if it's passed. It is omitted below.


In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV

xgb_pipeline = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", xgb.XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE)),
])

param_dist = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 6, 9],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__subsample": [0.7, 0.8, 0.9],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9],
}

random_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
random_search.fit(X_train_selected, y_train_e)

print(f"Best parameters: {random_search.best_params_}")
print(f"Best CV ROC-AUC:  {random_search.best_score_:.4f}")


In [ ]:
best_model = random_search.best_estimator_
y_pred_xgb = best_model.predict(X_test_selected)
y_proba_xgb = best_model.predict_proba(X_test_selected)[:, 1]

print(f"Tuned XGBoost -> Test ROC-AUC: {roc_auc_score(y_test_e, y_proba_xgb):.4f}")
print(f"Tuned XGBoost -> Test Recall:  {recall_score(y_test_e, y_pred_xgb):.4f}")


## 10. Final Comparison & Conclusion

In [ ]:
final_comparison = pd.DataFrame([
    {"Stage": "Baseline (best of 5 models)", "ROC-AUC": results_df["ROC-AUC"].max(),
     "Recall": results_df.loc[results_df["ROC-AUC"].idxmax(), "Recall"]},
    {"Stage": "SMOTE (best of 5 models)", "ROC-AUC": results_smote_df["ROC-AUC"].max(),
     "Recall": results_smote_df.loc[results_smote_df["ROC-AUC"].idxmax(), "Recall"]},
    {"Stage": "Undersampling (best of 5 models)", "ROC-AUC": results_under_df["ROC-AUC"].max(),
     "Recall": results_under_df.loc[results_under_df["ROC-AUC"].idxmax(), "Recall"]},
    {"Stage": "Feature-engineered Random Forest", "ROC-AUC": roc_auc_score(y_test_e, y_proba_eng),
     "Recall": recall_score(y_test_e, y_pred_eng)},
    {"Stage": "Tuned XGBoost", "ROC-AUC": roc_auc_score(y_test_e, y_proba_xgb),
     "Recall": recall_score(y_test_e, y_pred_xgb)},
])
final_comparison


### Conclusion

Across five baseline classifiers, two class-balancing strategies, engineered clinical
ratios, RFE-based feature selection, and a tuned XGBoost model, **ROC-AUC stays close to
0.50** — i.e., no configuration beats random guessing by a meaningful margin. Recall on the
Malignant class improves with SMOTE/undersampling, but that is purely a precision/recall
trade-off from rebalancing classes, not evidence of a learned clinical signal.

This lines up with the bivariate analysis in Section 4: none of the numerical or categorical
features showed a statistically significant relationship with `Biopsy_Result`, and group
statistics (e.g., mean PSA level, per-`Cancer_Stage` malignancy rate) were nearly identical
between classes. The most likely explanation is that `Biopsy_Result` in this file was
generated independently of the other columns — plausible for a synthetic/practice dataset,
but not something any amount of feature engineering or hyperparameter tuning can fix.

**Recommendations if this analysis is repeated on a real clinical dataset:**
1. Re-run Section 4's association tests first — if p-values stay uniformly high, revisit the
   data source/collection process before investing further in modeling.
2. If real predictive signal exists, prioritize `PSA_Level`, `DRE_Result`, `Age`, and
   `Prostate_Volume`, which are the clinically established predictors of prostate cancer risk.
3. Keep the leakage guard from Section 5 (excluding post-diagnosis columns such as
   `Cancer_Stage`/`Treatment_Recommended`) — those fields would not be available at
   prediction time in a real screening workflow.

### Summary of fixes applied to the original draft
| Issue in the draft | Fix in this version |
|---|---|
| ~70 cells including duplicated "task/reasoning" boilerplate and three near-identical copies of the model-training loop | Reduced to one linear pass per pipeline stage |
| Notebook re-read and re-encoded the CSV from scratch for the undersampling comparison | Reused the single `X_train_scaled`/`X_test_scaled` split throughout |
| `predict_proba(...)[:, 1]` assumed to be "Malignant" purely because of alphabetical class ordering | Target mapped to binary (`Malignant`=1) up front, so column 1 is guaranteed correct |
| `xgb.XGBClassifier(use_label_encoder=False, ...)` — removed argument in modern `xgboost`, raises `TypeError` | Argument removed |
| Comments/print statements mixed English and Vietnamese | Fully translated to English |
| Post-diagnosis columns (`Cancer_Stage`, `Treatment_Recommended`, `Survival_5_Years`, `Follow_Up_Required`) used as predictors of `Biopsy_Result` | Dropped before encoding (Section 5) to avoid target leakage |
